In [ ]:
import base64
import json
import requests
from datetime import date, timedelta

import polars as pl
import io

def generate_download_url(target_date):
    """
    Gera a URL para download do CSV da carteira do Ibovespa para uma data específica.
    """
    # A rota que você encontrou para download
    base_url = "https://sistemaswebb3-listados.b3.com.br/indexProxy/indexCall/GetDownloadPortfolioDay/"

    # Adicionamos o parâmetro 'refDate'
    params = {
        "index": "IBOV",
        "language": "pt-br",
        "refDate": target_date.strftime('%Y-%m-%d')
    }

    # Gera uma string JSON compacta (sem espaços)
    json_params = json.dumps(params, separators=(',', ':'))

    # Codifica para Base64
    base64_params = base64.b64encode(json_params.encode('utf-8')).decode('utf-8')

    return f"{base_url}{base64_params}"

def download_ibov_portfolio_csv(url, save_path):
    """
    Faz a requisição na URL e salva o conteúdo CSV em um arquivo.
    """
    print(f"Tentando fazer o download da URL: {url}")
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    try:
        res = requests.get(url, headers=headers, timeout=20)
        
        if res.status_code == 200:

            # 1. Decodificar o conteúdo usando 'latin-1' e transformá-lo em um "arquivo em memória"
            decoded_content = res.text.strip().encode('latin-1', 'replace').decode('utf-8', 'replace')
            content_as_file = io.StringIO(decoded_content)
            

            

            # # 2. Ler o CSV com Pandas, pulando linhas e definindo o separador
            # df = pl.read_csv(
            #     content_as_file,
            #     sep=';',            # Define o separador como ponto e vírgula
            #     skiprows=2,         # Pula as duas primeiras linhas de cabeçalho
            #     skipfooter=2,       # Pula as duas últimas linhas de rodapé
            #     engine='python',    # Usa o motor Python que é mais flexível com 'skipfooter'
            #     encoding='utf-8'    # O conteúdo já foi decodificado para utf-8
            # )

            # # 3. Renomear as colunas para nomes mais limpos
            # df.columns = [
            #     'ticker',
            #     'nome_ativo',
            #     'tipo_ativo',
            #     'quantidade_teorica',
            #     'participacao_percentual'
            # ]
            
            # # 4. Limpar os dados
            # df['tipo_ativo'] = df['tipo_ativo'].str.strip()
            # df['quantidade_teorica'] = df['quantidade_teorica'].str.replace('.', '', regex=False).astype('int64')
            # df['participacao_percentual'] = df['participacao_percentual'].str.replace(',', '.', regex=False).astype('float64')

            print("✅ Sucesso! Tabela limpa e organizada:")
            # print(df.head())



            # # Salva o conteúdo em um arquivo
            # with open(save_path, 'w', encoding='utf-8-sig') as f:
            #     f.write(res.text)
            # print(f"SUCESSO! Arquivo salvo em: {save_path}")
            
            # # Imprime as primeiras linhas para verificação
            # print("\n--- Início do Arquivo ---")
            # for line in res.text.splitlines()[:5]:
            #     print(line)
            # print("-------------------------")

        else:
            print(f"FALHA. Status: {res.status_code}, Content-Type: {res.headers.get('Content-Type')}")
            print("Resposta recebida:", res.text[:200]) # Mostra o início da resposta em caso de erro

    except requests.exceptions.RequestException as e:
        print(f"Erro de conexão: {e}")


# --- EXECUÇÃO ---

# 1. Escolha a data desejada
target_day = date(2025, 7, 1) # Uma sexta-feira, para garantir que é um dia de pregão

# 2. Gere a URL de download correta
download_url = generate_download_url(target_day)

# 3. Defina onde salvar o arquivo
file_name = f"IBOV_carteira_{target_day.strftime('%Y-%m-%d')}.csv"

# 4. Execute a função de download
download_ibov_portfolio_csv(download_url, file_name)

Tentando fazer o download da URL: https://sistemaswebb3-listados.b3.com.br/indexProxy/indexCall/GetDownloadPortfolioDay/eyJpbmRleCI6IklCT1YiLCJsYW5ndWFnZSI6InB0LWJyIiwicmVmRGF0ZSI6IjIwMjUtMDctMDEifQ==
✅ Sucesso! Tabela limpa e organizada:
